In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/bumba5341/advertisingcsv/Advertising.csv


In [2]:
data = pd.read_csv('/kaggle/input/datasets/bumba5341/advertisingcsv/Advertising.csv')

In [3]:
print(data.head())
print(data.info())

   Unnamed: 0     TV  Radio  Newspaper  Sales
0           1  230.1   37.8       69.2   22.1
1           2   44.5   39.3       45.1   10.4
2           3   17.2   45.9       69.3    9.3
3           4  151.5   41.3       58.5   18.5
4           5  180.8   10.8       58.4   12.9
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  200 non-null    int64  
 1   TV          200 non-null    float64
 2   Radio       200 non-null    float64
 3   Newspaper   200 non-null    float64
 4   Sales       200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB
None


In [4]:
data.isnull().sum()

Unnamed: 0    0
TV            0
Radio         0
Newspaper     0
Sales         0
dtype: int64

In [5]:
for col in data.select_dtypes(include='number').columns:
    data[col].fillna(data[col].median(), inplace=True)

for col in data.select_dtypes(include='object').columns:
    data[col].fillna(data[col].mode()[0], inplace=True)

/tmp/ipykernel_17/1689435564.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)


In [6]:
data["Total_Ad_Spend"] = data["TV"] + data["Radio"] + data["Newspaper"]

In [7]:
print(data.head())
print(data.info())

   Unnamed: 0     TV  Radio  Newspaper  Sales  Total_Ad_Spend
0           1  230.1   37.8       69.2   22.1           337.1
1           2   44.5   39.3       45.1   10.4           128.9
2           3   17.2   45.9       69.3    9.3           132.4
3           4  151.5   41.3       58.5   18.5           251.3
4           5  180.8   10.8       58.4   12.9           250.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      200 non-null    int64  
 1   TV              200 non-null    float64
 2   Radio           200 non-null    float64
 3   Newspaper       200 non-null    float64
 4   Sales           200 non-null    float64
 5   Total_Ad_Spend  200 non-null    float64
dtypes: float64(5), int64(1)
memory usage: 9.5 KB
None


In [8]:
from sklearn.model_selection import train_test_split

X = data.drop("Sales", axis=1)
y = data["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
data = pd.get_dummies(data, drop_first=True)

In [10]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 1.465060105010293
MSE: 3.199004468588908
R2: 0.8986489151417079


In [12]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Impact_on_Sales": model.coef_
})
print(coefficients.sort_values(by="Impact_on_Sales", ascending=False))

          Feature  Impact_on_Sales
2           Radio         0.129997
4  Total_Ad_Spend         0.059254
0      Unnamed: 0         0.000644
1              TV        -0.014535
3       Newspaper        -0.056208
